# MINERA Goiás — Motor Econômico-Energético V1

## Descrição do notebook e lógica do modelo

Este notebook apresenta uma versão demonstrativa e executável do modelo de projeção econômico-energética do setor mineral de Goiás para o período de 2027 a 2040.

Os dados desta versão são sintéticos (`estimated_demo`): eles não representam valores reais e servem para testar a estrutura do modelo, as regras dos cenários e os arquivos de saída. Posteriormente, os mesmos campos serão substituídos pelos dados reais do Squad 1 e pelos parâmetros produzidos pelos demais estudantes do Squad 2.

### Como o notebook funciona

1. **Carregamento dos dados**
   O notebook recebe e lê os arquivos de entrada: lista de minerais, histórico de produção, projetos futuros, intensidades energéticas, parâmetros dos cenários e regras de inclusão dos projetos.

2. **Construção do baseline de 2025**
   Para cada mineral, o modelo usa a produção observada em 2025 como ponto de partida. A intensidade energética é calculada de forma ponderada pela produção das operações, mantendo sempre a mesma `production_basis` — por exemplo, produção beneficiada ou conteúdo mineral.

3. **Projeção da produção existente**
   A produção das operações já existentes é projetada anualmente de 2027 a 2040. Em cada cenário, essa projeção usa uma taxa de crescimento anual externa ao código (`existing_production_growth_rate`), permitindo alterar as hipóteses sem modificar a lógica do motor.

4. **Incorporação dos projetos futuros**
   Cada projeto possui mineral, capacidade anual, ano previsto de início e estágio de maturidade. As regras dos cenários definem quais projetos entram no cálculo e se há atraso no início. Quando ativo, um projeto adiciona produção conforme sua capacidade, taxa de utilização e fator de captura de mercado.

5. **Produção total projetada**
   Para cada mineral, ano e cenário, a produção total é dada por:

   `produção projetada = produção existente + produção dos projetos`

6. **Cálculo da demanda de energia**
   A intensidade energética pode melhorar ao longo do tempo segundo uma hipótese de eficiência anual parametrizável. A demanda de energia é calculada de forma direta e auditável:

   `demanda de energia (MWh) = produção projetada (t) × intensidade energética (MWh/t)`

7. **Agregação para Goiás**
   Após calcular os resultados por mineral, o notebook soma a produção e a demanda de energia para obter o total anual do estado de Goiás em cada cenário.

8. **Verificações de consistência**
   Antes da exportação, o modelo verifica se todos os minerais possuem os 14 anos e os 3 cenários, se não há valores ausentes ou negativos e se o total de Goiás é exatamente igual à soma dos minerais.

9. **Exportação dos resultados**
   O notebook gera três arquivos: projeção por mineral, total anual de Goiás e contribuição de cada projeto. Todos os resultados demo são identificados como `estimated_demo` e mantêm um `source_id` para rastreabilidade.


In [1]:
from google.colab import files

uploaded = files.upload()

Saving MINERA_Goias_dados_demo_v2.zip to MINERA_Goias_dados_demo_v2.zip


In [2]:
import zipfile
import os

with zipfile.ZipFile("MINERA_Goias_dados_demo_v2.zip", "r") as zip_file:
    zip_file.extractall("data_demo")

print("File disponibili:")
print(os.listdir("data_demo"))

File disponibili:
['README.md', 'minerals_demo.csv', 'sources_demo.csv', 'scenario_parameters_demo.csv', 'production_history_demo.csv', 'scenario_project_rules_demo.csv', 'projects_demo.csv', 'energy_intensity_demo.csv']


In [3]:
import pandas as pd

minerals = pd.read_csv("data_demo/minerals_demo.csv", encoding="utf-8-sig")
production = pd.read_csv("data_demo/production_history_demo.csv", encoding="utf-8-sig")
projects = pd.read_csv("data_demo/projects_demo.csv", encoding="utf-8-sig")
intensity = pd.read_csv("data_demo/energy_intensity_demo.csv", encoding="utf-8-sig")
parameters = pd.read_csv("data_demo/scenario_parameters_demo.csv", encoding="utf-8-sig")
project_rules = pd.read_csv("data_demo/scenario_project_rules_demo.csv", encoding="utf-8-sig")

print("Minerali:", len(minerals))
print("Righe di produzione storica:", len(production))
print("Progetti:", len(projects))
print("Righe di intensità:", len(intensity))
print("Parametri di scenario:", len(parameters))

display(production.head())
display(projects.head())

Minerali: 8
Righe di produzione storica: 80
Progetti: 10
Righe di intensità: 30
Parametri di scenario: 1344


,mineral_id,company_id,operation_id,year,production_t,production_basis,data_nature,source_id
0,MIN_001,COM_DEMO_001,OP_DEMO_NI_001,2018,19394.906,beneficiada,estimated_demo,SRC_DEMO_001
1,MIN_001,COM_DEMO_001,OP_DEMO_NI_001,2019,19883.689,beneficiada,estimated_demo,SRC_DEMO_001
2,MIN_001,COM_DEMO_001,OP_DEMO_NI_001,2020,20383.485,beneficiada,estimated_demo,SRC_DEMO_001
3,MIN_001,COM_DEMO_001,OP_DEMO_NI_001,2021,20400.957,beneficiada,estimated_demo,SRC_DEMO_001
4,MIN_001,COM_DEMO_001,OP_DEMO_NI_001,2022,20915.094,beneficiada,estimated_demo,SRC_DEMO_001


,project_id,mineral_id,company_id,capacity_tpy,start_year,project_stage,production_basis,data_nature,source_id
0,PRJ_DEMO_NI_001,MIN_001,COM_DEMO_001,12000.0,2028,definido,beneficiada,estimated_demo,SRC_DEMO_001
1,PRJ_DEMO_NI_002,MIN_001,COM_DEMO_011,18000.0,2032,provavel,beneficiada,estimated_demo,SRC_DEMO_001
2,PRJ_DEMO_CU_001,MIN_002,COM_DEMO_003,15000.0,2029,definido,conteudo_mineral,estimated_demo,SRC_DEMO_001
3,PRJ_DEMO_CU_002,MIN_002,COM_DEMO_012,22000.0,2034,possivel,conteudo_mineral,estimated_demo,SRC_DEMO_001
4,PRJ_DEMO_NB_001,MIN_003,COM_DEMO_005,14000.0,2030,definido,beneficiada,estimated_demo,SRC_DEMO_001


In [4]:
production_2025 = production[production["year"] == 2025].copy()
intensity_2025 = intensity[intensity["year"] == 2025].copy()

baseline_detail = production_2025.merge(
    intensity_2025[
        ["mineral_id", "operation_id", "production_basis", "energy_intensity_mwh_t"]
    ],
    on=["mineral_id", "operation_id", "production_basis"],
    how="inner"
)

baseline_detail["energy_mwh"] = (
    baseline_detail["production_t"] * baseline_detail["energy_intensity_mwh_t"]
)

baseline = (
    baseline_detail
    .groupby(["mineral_id", "production_basis"], as_index=False)
    .agg(
        baseline_production_t=("production_t", "sum"),
        baseline_energy_mwh=("energy_mwh", "sum")
    )
)

baseline["baseline_intensity_mwh_t"] = (
    baseline["baseline_energy_mwh"] / baseline["baseline_production_t"]
)

baseline = baseline.merge(
    minerals[["mineral_id", "mineral_name"]],
    on="mineral_id",
    how="left"
)

display(
    baseline[
        [
            "mineral_id",
            "mineral_name",
            "production_basis",
            "baseline_production_t",
            "baseline_intensity_mwh_t"
        ]
    ]
)


,mineral_id,mineral_name,production_basis,baseline_production_t,baseline_intensity_mwh_t
0,MIN_001,Níquel,beneficiada,35608.0,4.891081
1,MIN_002,Cobre,conteudo_mineral,50648.0,3.284094
2,MIN_003,Nióbio,beneficiada,56000.0,2.200000
3,MIN_004,Fosfato,beneficiada,1600000.0,0.420000
4,MIN_005,Minério de alumínio,beneficiada,920000.0,0.580000
5,MIN_006,Ouro,conteudo_mineral,5.5,28.000000
6,MIN_007,Amianto,beneficiada,180000.0,0.950000
7,MIN_008,Apatita,beneficiada,900000.0,0.500000


In [6]:
growth_parameters = parameters[
    parameters["parameter_name"] == "existing_production_growth_rate"
].copy()

existing_projection = baseline.merge(
    growth_parameters[
        ["mineral_id", "year", "scenario", "parameter_value"]
    ],
    on="mineral_id",
    how="inner"
)

existing_projection = existing_projection.sort_values(
    ["mineral_id", "scenario", "year"]
).copy()

# Nel 2027 applichiamo il tasso due volte: 2025→2026 e 2026→2027.
existing_projection["annual_growth_factor"] = (
    1 + existing_projection["parameter_value"]
)

existing_projection["growth_factor"] = existing_projection[
    "annual_growth_factor"
]

existing_projection.loc[
    existing_projection["year"] == 2027,
    "growth_factor"
] = existing_projection.loc[
    existing_projection["year"] == 2027,
    "annual_growth_factor"
] ** 2

existing_projection["cumulative_growth_factor"] = (
    existing_projection
    .groupby(["mineral_id", "scenario"])["growth_factor"]
    .cumprod()
)

existing_projection["existing_production_t"] = (
    existing_projection["baseline_production_t"]
    * existing_projection["cumulative_growth_factor"]
)

display(
    existing_projection[
        existing_projection["year"].isin([2027, 2040])
    ][
        [
            "mineral_id",
            "mineral_name",
            "year",
            "scenario",
            "existing_production_t"
        ]
    ]
)

,mineral_id,mineral_name,year,scenario,existing_production_t
0,MIN_001,Níquel,2027,conservador,3.580626e+04
39,MIN_001,Níquel,2040,conservador,3.712210e+04
2,MIN_001,Níquel,2027,expansao,3.703058e+04
41,MIN_001,Níquel,2040,expansao,4.776887e+04
1,MIN_001,Níquel,2027,referencia,3.637913e+04
40,MIN_001,Níquel,2040,referencia,4.181506e+04
42,MIN_002,Cobre,2027,conservador,5.096555e+04
81,MIN_002,Cobre,2040,conservador,5.307874e+04
44,MIN_002,Cobre,2027,expansao,5.270761e+04
83,MIN_002,Cobre,2040,expansao,6.829597e+04


In [7]:
# Prendiamo dai parametri solo il tasso di utilizzo dei progetti.
utilization_parameters = parameters[
    parameters["parameter_name"] == "project_utilization_rate"
][
    ["mineral_id", "year", "scenario", "parameter_value"]
].rename(columns={"parameter_value": "project_utilization_rate"})

# Rendiamo esplicito il campo vero/falso nella tabella delle regole.
project_rules["include_project"] = (
    project_rules["include_project"]
    .astype(str)
    .str.lower()
    .eq("true")
)

# Creiamo una riga per ogni progetto, anno e scenario.
project_calculation = (
    existing_projection[
        ["mineral_id", "year", "scenario"]
    ]
    .merge(
        projects[
            [
                "project_id",
                "mineral_id",
                "capacity_tpy",
                "start_year",
                "project_stage",
                "production_basis"
            ]
        ],
        on="mineral_id",
        how="left"
    )
    .merge(
        project_rules[
            [
                "scenario",
                "project_stage",
                "include_project",
                "start_delay_years"
            ]
        ],
        on=["scenario", "project_stage"],
        how="left"
    )
    .merge(
        utilization_parameters,
        on=["mineral_id", "year", "scenario"],
        how="left"
    )
)

# Un progetto produce solo dopo l'anno di avvio, più l'eventuale ritardo.
project_calculation["effective_start_year"] = (
    project_calculation["start_year"]
    + project_calculation["start_delay_years"]
)

project_calculation["project_production_t"] = 0.0

active_project = (
    project_calculation["include_project"]
    & (project_calculation["year"] >= project_calculation["effective_start_year"])
)

project_calculation.loc[active_project, "project_production_t"] = (
    project_calculation.loc[active_project, "capacity_tpy"]
    * project_calculation.loc[active_project, "project_utilization_rate"]
)

# Sommiamo tutti i progetti dello stesso minerale, anno e scenario.
project_total = (
    project_calculation
    .groupby(["mineral_id", "year", "scenario"], as_index=False)
    .agg(project_production_t=("project_production_t", "sum"))
)

# Produzione totale = attività esistenti + progetti entrati in funzione.
production_projection = existing_projection.merge(
    project_total,
    on=["mineral_id", "year", "scenario"],
    how="left"
)

production_projection["project_production_t"] = (
    production_projection["project_production_t"].fillna(0)
)

production_projection["projected_production_t"] = (
    production_projection["existing_production_t"]
    + production_projection["project_production_t"]
)

display(
    production_projection[
        production_projection["year"].isin([2027, 2030, 2040])
    ][
        [
            "mineral_id",
            "mineral_name",
            "year",
            "scenario",
            "existing_production_t",
            "project_production_t",
            "projected_production_t"
        ]
    ]
)

,mineral_id,mineral_name,year,scenario,existing_production_t,project_production_t,projected_production_t
0,MIN_001,Níquel,2027,conservador,3.580626e+04,0.0,3.580626e+04
3,MIN_001,Níquel,2030,conservador,3.610571e+04,7548.0,4.365371e+04
13,MIN_001,Níquel,2040,conservador,3.712210e+04,7908.0,4.503010e+04
14,MIN_001,Níquel,2027,expansao,3.703058e+04,0.0,3.703058e+04
17,MIN_001,Níquel,2030,expansao,3.927173e+04,10908.0,5.017973e+04
...,...,...,...,...,...,...,...
311,MIN_008,Apatita,2030,expansao,1.004583e+06,0.0,1.004583e+06
321,MIN_008,Apatita,2040,expansao,1.251618e+06,262920.0,1.514538e+06
322,MIN_008,Apatita,2027,referencia,9.239715e+05,0.0,9.239715e+05
325,MIN_008,Apatita,2030,referencia,9.611313e+05,0.0,9.611313e+05


In [8]:
market_capture_parameters = parameters[
    parameters["parameter_name"] == "market_capture_factor"
][
    ["mineral_id", "year", "scenario", "parameter_value"]
].rename(columns={"parameter_value": "market_capture_factor"})

# Aggiungiamo il fattore di mercato al calcolo già creato.
project_calculation = project_calculation.merge(
    market_capture_parameters,
    on=["mineral_id", "year", "scenario"],
    how="left"
)

# Ricalcoliamo la produzione dei progetti.
project_calculation["project_production_t"] = 0.0

active_project = (
    project_calculation["include_project"]
    & (project_calculation["year"] >= project_calculation["effective_start_year"])
)

project_calculation.loc[active_project, "project_production_t"] = (
    project_calculation.loc[active_project, "capacity_tpy"]
    * project_calculation.loc[active_project, "project_utilization_rate"]
    * project_calculation.loc[active_project, "market_capture_factor"]
)

project_total = (
    project_calculation
    .groupby(["mineral_id", "year", "scenario"], as_index=False)
    .agg(project_production_t=("project_production_t", "sum"))
)

production_projection = existing_projection.merge(
    project_total,
    on=["mineral_id", "year", "scenario"],
    how="left"
)

production_projection["project_production_t"] = (
    production_projection["project_production_t"].fillna(0)
)

production_projection["projected_production_t"] = (
    production_projection["existing_production_t"]
    + production_projection["project_production_t"]
)

display(
    production_projection[
        (production_projection["mineral_id"] == "MIN_001")
        & (production_projection["year"].isin([2027, 2030, 2040]))
    ][
        [
            "mineral_name",
            "year",
            "scenario",
            "existing_production_t",
            "project_production_t",
            "projected_production_t"
        ]
    ]
)

,mineral_name,year,scenario,existing_production_t,project_production_t,projected_production_t
0,Níquel,2027,conservador,35806.255673,0.000,35806.255673
3,Níquel,2030,conservador,36105.710790,7193.244,43298.954790
13,Níquel,2040,conservador,37122.099863,7615.404,44737.503863
14,Níquel,2027,expansao,37030.584053,0.000,37030.584053
17,Níquel,2030,expansao,39271.729957,11595.204,50866.933957
27,Níquel,2040,expansao,47768.866464,30226.410,77995.276464
28,Níquel,2027,referencia,36379.126595,0.000,36379.126595
31,Níquel,2030,referencia,37567.240783,9737.124,47304.364783
41,Níquel,2040,referencia,41815.060813,25497.210,67312.270813


In [9]:
efficiency_parameters = parameters[
    parameters["parameter_name"] == "annual_efficiency_improvement_rate"
][
    ["mineral_id", "year", "scenario", "parameter_value"]
].rename(columns={"parameter_value": "annual_efficiency_improvement_rate"})

energy_projection = production_projection.merge(
    efficiency_parameters,
    on=["mineral_id", "year", "scenario"],
    how="left"
)

energy_projection = energy_projection.sort_values(
    ["mineral_id", "scenario", "year"]
).copy()

# L'intensità diminuisce grazie all'efficienza.
energy_projection["annual_efficiency_factor"] = (
    1 - energy_projection["annual_efficiency_improvement_rate"]
)

energy_projection["efficiency_factor"] = (
    energy_projection["annual_efficiency_factor"]
)

# Dal 2025 al 2027 consideriamo due anni di miglioramento.
energy_projection.loc[
    energy_projection["year"] == 2027,
    "efficiency_factor"
] = energy_projection.loc[
    energy_projection["year"] == 2027,
    "annual_efficiency_factor"
] ** 2

energy_projection["cumulative_efficiency_factor"] = (
    energy_projection
    .groupby(["mineral_id", "scenario"])["efficiency_factor"]
    .cumprod()
)

energy_projection["energy_intensity_mwh_t"] = (
    energy_projection["baseline_intensity_mwh_t"]
    * energy_projection["cumulative_efficiency_factor"]
)

energy_projection["energy_demand_mwh"] = (
    energy_projection["projected_production_t"]
    * energy_projection["energy_intensity_mwh_t"]
)

final_output = energy_projection[
    [
        "mineral_id",
        "mineral_name",
        "production_basis",
        "year",
        "scenario",
        "projected_production_t",
        "energy_intensity_mwh_t",
        "energy_demand_mwh"
    ]
].copy()

display(
    final_output[
        (final_output["mineral_id"] == "MIN_001")
        & (final_output["year"].isin([2027, 2030, 2040]))
    ]
)

,mineral_id,mineral_name,production_basis,year,scenario,projected_production_t,energy_intensity_mwh_t,energy_demand_mwh
0,MIN_001,Níquel,beneficiada,2027,conservador,35806.255673,4.852030,173733.036311
3,MIN_001,Níquel,beneficiada,2030,conservador,43298.954790,4.794038,207576.855916
13,MIN_001,Níquel,beneficiada,2040,conservador,44737.503863,4.605692,206047.168033
14,MIN_001,Níquel,beneficiada,2027,expansao,37030.584053,4.755089,176083.724726
17,MIN_001,Níquel,beneficiada,2030,expansao,50866.933957,4.558158,231859.534860
27,MIN_001,Níquel,beneficiada,2040,expansao,77995.276464,3.958754,308764.136502
28,MIN_001,Níquel,beneficiada,2027,referencia,36379.126595,4.803437,174744.856603
31,MIN_001,Níquel,beneficiada,2030,referencia,47304.364783,4.674908,221143.567836
41,MIN_001,Níquel,beneficiada,2040,referencia,67312.270813,4.270804,287477.516134


In [11]:
goias_total = (
    final_output
    .groupby(["year", "scenario"], as_index=False)
    .agg(
        total_projected_production_t=("projected_production_t", "sum"),
        total_energy_demand_mwh=("energy_demand_mwh", "sum")
    )
)

goias_total["state_energy_intensity_mwh_t"] = (
    goias_total["total_energy_demand_mwh"]
    / goias_total["total_projected_production_t"]
)

display(goias_total)

,year,scenario,total_projected_production_t,total_energy_demand_mwh,state_energy_intensity_mwh_t
0,2027,conservador,3.774332e+06,2.290889e+06,0.606966
1,2027,expansao,3.903184e+06,2.321784e+06,0.594844
2,2027,referencia,3.834724e+06,2.304236e+06,0.600887
3,2028,conservador,3.790472e+06,2.291112e+06,0.604440
4,2028,expansao,4.333051e+06,2.577954e+06,0.594951
5,2028,referencia,4.172788e+06,2.515884e+06,0.602926
6,2029,conservador,4.022418e+06,2.444812e+06,0.607797
7,2029,expansao,4.433772e+06,2.636339e+06,0.594604
8,2029,referencia,4.233947e+06,2.560323e+06,0.604713
9,2030,conservador,4.048953e+06,2.474260e+06,0.611086


In [12]:
expected_years = set(range(2027, 2041))
expected_scenarios = {"conservador", "referencia", "expansao"}

# Ogni combinazione minerale-scenario deve avere tutti i 14 anni.
coverage = (
    final_output
    .groupby(["mineral_id", "scenario"])["year"]
    .agg(["count", "min", "max"])
    .reset_index()
)

all_years_present = (
    (coverage["count"] == 14).all()
    and (coverage["min"] == 2027).all()
    and (coverage["max"] == 2040).all()
)

# Il totale Goiás deve coincidere esattamente con la somma dei minerali.
recalculated_total = (
    final_output
    .groupby(["year", "scenario"], as_index=False)
    .agg(
        production_from_minerals=("projected_production_t", "sum"),
        energy_from_minerals=("energy_demand_mwh", "sum")
    )
)

reconciliation = goias_total.merge(
    recalculated_total,
    on=["year", "scenario"],
    how="left"
)

reconciliation["production_difference"] = (
    reconciliation["total_projected_production_t"]
    - reconciliation["production_from_minerals"]
)

reconciliation["energy_difference"] = (
    reconciliation["total_energy_demand_mwh"]
    - reconciliation["energy_from_minerals"]
)

check_results = pd.DataFrame({
    "controle": [
        "Número de linhas detalhadas",
        "Número de linhas totais Goiás",
        "Anos completos para cada mineral e cenário",
        "Cenários corretos",
        "Valores ausentes",
        "Produção e energia sempre positivas",
        "Total Goiás = soma dos minerais"
    ],
    "resultado": [
        len(final_output) == 8 * 14 * 3,
        len(goias_total) == 14 * 3,
        all_years_present,
        set(final_output["scenario"]) == expected_scenarios,
        final_output.isna().sum().sum() == 0,
        (
            (final_output["projected_production_t"] > 0).all()
            and (final_output["energy_intensity_mwh_t"] > 0).all()
            and (final_output["energy_demand_mwh"] > 0).all()
        ),
        (
            reconciliation["production_difference"].abs().max() < 0.0001
            and reconciliation["energy_difference"].abs().max() < 0.0001
        )
    ]
})

display(check_results)
display(reconciliation)

,controle,resultado
0,Número de linhas detalhadas,True
1,Número de linhas totais Goiás,True
2,Anos completos para cada mineral e cenário,True
3,Cenários corretos,True
4,Valores ausentes,True
5,Produção e energia sempre positivas,True
6,Total Goiás = soma dos minerais,True


,year,scenario,total_projected_production_t,total_energy_demand_mwh,state_energy_intensity_mwh_t,production_from_minerals,energy_from_minerals,production_difference,energy_difference
0,2027,conservador,3.774332e+06,2.290889e+06,0.606966,3.774332e+06,2.290889e+06,0.0,0.0
1,2027,expansao,3.903184e+06,2.321784e+06,0.594844,3.903184e+06,2.321784e+06,0.0,0.0
2,2027,referencia,3.834724e+06,2.304236e+06,0.600887,3.834724e+06,2.304236e+06,0.0,0.0
3,2028,conservador,3.790472e+06,2.291112e+06,0.604440,3.790472e+06,2.291112e+06,0.0,0.0
4,2028,expansao,4.333051e+06,2.577954e+06,0.594951,4.333051e+06,2.577954e+06,0.0,0.0
5,2028,referencia,4.172788e+06,2.515884e+06,0.602926,4.172788e+06,2.515884e+06,0.0,0.0
6,2029,conservador,4.022418e+06,2.444812e+06,0.607797,4.022418e+06,2.444812e+06,0.0,0.0
7,2029,expansao,4.433772e+06,2.636339e+06,0.594604,4.433772e+06,2.636339e+06,0.0,0.0
8,2029,referencia,4.233947e+06,2.560323e+06,0.604713,4.233947e+06,2.560323e+06,0.0,0.0
9,2030,conservador,4.048953e+06,2.474260e+06,0.611086,4.048953e+06,2.474260e+06,0.0,0.0


In [17]:
# ANÁLISE DE SENSIBILIDADE PARAMETRIZÁVEL
# Esta célula mantém o cenário de referência e altera uma hipótese por vez.
# Os intervalos podem ser modificados sem mudar a lógica do modelo.

scenario_base = "referencia"

# Atrasos adicionais: 0 a 24 meses, em passos de 6 meses.
delay_months_values = list(range(0, 25, 6))

# Variação geral da utilização: -20 a +10 p.p., em passos de 1 p.p.
# A utilização final é sempre limitada entre 0% e 100%.
utilization_adjustment_pp_values = list(range(-20, 11, 1))

# Variação do ganho anual de eficiência: -2,0 a +2,0 p.p.,
# em passos de 0,1 p.p.
# Valor positivo = maior redução anual da intensidade energética.
efficiency_adjustment_pp_values = [
    round(value / 10, 1) for value in range(-20, 21)
]

# Resultado original do cenário de referência.
reference_output = final_output[
    final_output["scenario"] == scenario_base
].copy()

# Produção das operações existentes no cenário de referência.
reference_existing = production_projection[
    production_projection["scenario"] == scenario_base
][
    ["mineral_id", "year", "existing_production_t"]
].copy()

# Detalhe dos projetos no cenário de referência.
reference_projects = project_calculation[
    project_calculation["scenario"] == scenario_base
].copy()

# Intensidade energética original para testes que não alteram eficiência.
reference_intensity = reference_output[
    [
        "mineral_id",
        "year",
        "mineral_name",
        "production_basis",
        "energy_intensity_mwh_t"
    ]
].copy()


def calculate_project_sensitivity(
    test_group,
    parameter_name,
    parameter_value,
    additional_delay_months=0,
    utilization_adjustment_pp=0
):
    """
    Recalcula a produção dos projetos.
    A produção existente e a intensidade energética permanecem iguais
    ao cenário de referência.
    """

    test = reference_projects.copy()

    # Converte o atraso adicional em anos completos e meses restantes.
    full_delay_years = additional_delay_months // 12
    remaining_delay_months = additional_delay_months % 12

    # Ano de início após o atraso adicional.
    test["effective_start_year_test"] = (
        test["effective_start_year"] + full_delay_years
    )

    # Aplica variação geral à utilização e limita o resultado entre 0% e 100%.
    test["project_utilization_rate_test"] = (
        test["project_utilization_rate"]
        + utilization_adjustment_pp / 100
    ).clip(lower=0, upper=1)

    # Um projeto produz somente após o ano efetivo de entrada.
    project_active = (
        test["include_project"]
        & (test["year"] >= test["effective_start_year_test"])
    )

    test["operation_share"] = 0.0
    test.loc[project_active, "operation_share"] = 1.0

    # Atrasos de 6 ou 18 meses geram produção parcial no primeiro ano ativo.
    if remaining_delay_months > 0:
        partial_first_year = (
            project_active
            & (test["year"] == test["effective_start_year_test"])
        )

        test.loc[partial_first_year, "operation_share"] = (
            1 - remaining_delay_months / 12
        )

    # Produção = capacidade × utilização × captura de mercado
    # × parcela do ano em operação.
    test["project_production_t_test"] = (
        test["capacity_tpy"]
        * test["project_utilization_rate_test"]
        * test["market_capture_factor"]
        * test["operation_share"]
    )

    project_total = (
        test
        .groupby(["mineral_id", "year"], as_index=False)
        .agg(project_production_t=("project_production_t_test", "sum"))
    )

    result = reference_existing.merge(
        project_total,
        on=["mineral_id", "year"],
        how="left"
    ).merge(
        reference_intensity,
        on=["mineral_id", "year"],
        how="left"
    )

    result["project_production_t"] = (
        result["project_production_t"].fillna(0)
    )

    result["projected_production_t"] = (
        result["existing_production_t"]
        + result["project_production_t"]
    )

    result["energy_demand_mwh"] = (
        result["projected_production_t"]
        * result["energy_intensity_mwh_t"]
    )

    result["test_group"] = test_group
    result["parameter_name"] = parameter_name
    result["parameter_value"] = parameter_value

    return result


# 1. Sensibilidade ao atraso dos projetos.
delay_results = []

for delay_months in delay_months_values:
    delay_results.append(
        calculate_project_sensitivity(
            test_group="atraso_de_projetos",
            parameter_name="atraso_adicional_meses",
            parameter_value=delay_months,
            additional_delay_months=delay_months
        )
    )

delay_results = pd.concat(delay_results, ignore_index=True)


# 2. Sensibilidade à utilização geral dos projetos.
utilization_results = []

for utilization_adjustment_pp in utilization_adjustment_pp_values:
    utilization_results.append(
        calculate_project_sensitivity(
            test_group="utilizacao_dos_projetos",
            parameter_name="variacao_utilizacao_pp",
            parameter_value=utilization_adjustment_pp,
            utilization_adjustment_pp=utilization_adjustment_pp
        )
    )

utilization_results = pd.concat(utilization_results, ignore_index=True)


# 3. Sensibilidade à eficiência energética.
# A produção permanece igual; mudam a intensidade e a demanda de energia.
efficiency_parameters = parameters[
    (parameters["scenario"] == scenario_base)
    & (parameters["parameter_name"] == "annual_efficiency_improvement_rate")
][
    ["mineral_id", "year", "parameter_value"]
].rename(columns={"parameter_value": "efficiency_rate"})

efficiency_results = []

for efficiency_adjustment_pp in efficiency_adjustment_pp_values:

    test = reference_output[
        [
            "mineral_id",
            "mineral_name",
            "production_basis",
            "year",
            "projected_production_t"
        ]
    ].copy()

    test = test.merge(
        baseline[
            [
                "mineral_id",
                "production_basis",
                "baseline_intensity_mwh_t"
            ]
        ],
        on=["mineral_id", "production_basis"],
        how="left"
    ).merge(
        efficiency_parameters,
        on=["mineral_id", "year"],
        how="left"
    ).sort_values(["mineral_id", "year"])

    # A taxa original representa redução da intensidade energética.
    # Valor positivo no teste aumenta essa redução; valor negativo a diminui.
    test["annual_efficiency_factor"] = (
        1
        - test["efficiency_rate"]
        - efficiency_adjustment_pp / 100
    )

    test["efficiency_factor"] = test["annual_efficiency_factor"]

    # Em 2027 o fator é aplicado duas vezes: 2025→2026 e 2026→2027.
    test.loc[
        test["year"] == 2027,
        "efficiency_factor"
    ] = test.loc[
        test["year"] == 2027,
        "annual_efficiency_factor"
    ] ** 2

    test["cumulative_efficiency_factor"] = (
        test
        .groupby("mineral_id")["efficiency_factor"]
        .cumprod()
    )

    test["energy_intensity_mwh_t"] = (
        test["baseline_intensity_mwh_t"]
        * test["cumulative_efficiency_factor"]
    )

    test["energy_demand_mwh"] = (
        test["projected_production_t"]
        * test["energy_intensity_mwh_t"]
    )

    test["test_group"] = "eficiencia_energetica"
    test["parameter_name"] = "variacao_melhoria_eficiencia_pp"
    test["parameter_value"] = efficiency_adjustment_pp

    efficiency_results.append(test)

efficiency_results = pd.concat(efficiency_results, ignore_index=True)


# Junta todas as análises: mineral × ano × valor do parâmetro.
sensitivity_results = pd.concat(
    [
        delay_results,
        utilization_results,
        efficiency_results
    ],
    ignore_index=True
)

# Agrega cada teste para o total anual de Goiás.
sensitivity_goias_total = (
    sensitivity_results
    .groupby(
        ["test_group", "parameter_name", "parameter_value", "year"],
        as_index=False
    )
    .agg(
        projected_production_t=("projected_production_t", "sum"),
        energy_demand_mwh=("energy_demand_mwh", "sum")
    )
)

# Adiciona o cenário de referência para calcular diferenças.
reference_goias_total = goias_total[
    goias_total["scenario"] == scenario_base
][
    ["year", "total_projected_production_t", "total_energy_demand_mwh"]
].copy()

sensitivity_goias_total = sensitivity_goias_total.merge(
    reference_goias_total,
    on="year",
    how="left"
)

sensitivity_goias_total["production_difference_t"] = (
    sensitivity_goias_total["projected_production_t"]
    - sensitivity_goias_total["total_projected_production_t"]
)

sensitivity_goias_total["energy_difference_mwh"] = (
    sensitivity_goias_total["energy_demand_mwh"]
    - sensitivity_goias_total["total_energy_demand_mwh"]
)

sensitivity_goias_total["production_difference_pct"] = (
    100
    * sensitivity_goias_total["production_difference_t"]
    / sensitivity_goias_total["total_projected_production_t"]
)

sensitivity_goias_total["energy_difference_pct"] = (
    100
    * sensitivity_goias_total["energy_difference_mwh"]
    / sensitivity_goias_total["total_energy_demand_mwh"]
)

# Mostra a tabela final para os anos-chave.
sensitivity_summary = sensitivity_goias_total[
    sensitivity_goias_total["year"].isin([2030, 2035, 2040])
].sort_values(
    ["test_group", "parameter_value", "year"]
).copy()

display(sensitivity_summary.round(2))

,test_group,parameter_name,parameter_value,year,projected_production_t,energy_demand_mwh,total_projected_production_t,total_energy_demand_mwh,production_difference_t,energy_difference_mwh,production_difference_pct,energy_difference_pct
3,atraso_de_projetos,atraso_adicional_meses,0.0,2030,4359912.78,2649077.71,4359912.78,2649077.71,0.00,0.00,0.00,0.00
8,atraso_de_projetos,atraso_adicional_meses,0.0,2035,5240546.08,2993708.56,5240546.08,2993708.56,0.00,0.00,0.00,0.00
13,atraso_de_projetos,atraso_adicional_meses,0.0,2040,5530209.03,3015545.02,5530209.03,3015545.02,0.00,0.00,0.00,0.00
17,atraso_de_projetos,atraso_adicional_meses,6.0,2030,4321775.71,2607662.59,4359912.78,2649077.71,-38137.07,-41415.13,-0.87,-1.56
22,atraso_de_projetos,atraso_adicional_meses,6.0,2035,5240546.08,2993708.56,5240546.08,2993708.56,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
1058,utilizacao_dos_projetos,variacao_utilizacao_pp,9.0,2035,5351133.76,3062894.65,5240546.08,2993708.56,110587.68,69186.09,2.11,2.31
1063,utilizacao_dos_projetos,variacao_utilizacao_pp,9.0,2040,5641345.26,3082001.29,5530209.03,3015545.02,111136.23,66456.28,2.01,2.20
1067,utilizacao_dos_projetos,variacao_utilizacao_pp,10.0,2030,4407154.08,2689126.58,4359912.78,2649077.71,47241.30,40048.87,1.08,1.51
1072,utilizacao_dos_projetos,variacao_utilizacao_pp,10.0,2035,5363421.28,3070581.99,5240546.08,2993708.56,122875.20,76873.44,2.34,2.57


In [18]:
import os
import shutil
from google.colab import files

# Cria a pasta que receberá todos os outputs do modelo.
os.makedirs("outputs", exist_ok=True)

# 1. Output principal: produção e demanda de energia por mineral, ano e cenário.
output_by_mineral = final_output.copy()
output_by_mineral["data_nature"] = "estimated_demo"
output_by_mineral["source_id"] = "SRC_DEMO_001"

# 2. Output agregado: total anual de Goiás por cenário.
output_goias_total = goias_total.copy()
output_goias_total["data_nature"] = "estimated_demo"
output_goias_total["source_id"] = "SRC_DEMO_001"

# 3. Output de rastreabilidade: contribuição de cada projeto.
output_by_project = project_calculation[
    [
        "project_id",
        "mineral_id",
        "year",
        "scenario",
        "project_stage",
        "start_year",
        "effective_start_year",
        "capacity_tpy",
        "production_basis",
        "project_utilization_rate",
        "market_capture_factor",
        "project_production_t"
    ]
].copy()

output_by_project["data_nature"] = "estimated_demo"
output_by_project["source_id"] = "SRC_DEMO_001"

# 4. Sensibilidade detalhada:
# um registro por mineral, ano e valor testado de cada parâmetro.
output_sensitivity_detail = sensitivity_results.copy()
output_sensitivity_detail["data_nature"] = "estimated_demo"
output_sensitivity_detail["source_id"] = "SRC_DEMO_001"

# 5. Sensibilidade agregada:
# total anual de Goiás para todos os valores testados.
output_sensitivity_goias = sensitivity_goias_total.copy()
output_sensitivity_goias["data_nature"] = "estimated_demo"
output_sensitivity_goias["source_id"] = "SRC_DEMO_001"

# 6. Resumo da sensibilidade:
# apenas os anos-chave 2030, 2035 e 2040.
output_sensitivity_summary = sensitivity_summary.copy()
output_sensitivity_summary["data_nature"] = "estimated_demo"
output_sensitivity_summary["source_id"] = "SRC_DEMO_001"

# Salva os arquivos em UTF-8 com BOM para preservar acentos no Excel.
output_by_mineral.to_csv(
    "outputs/projecao_por_mineral_demo.csv",
    index=False,
    encoding="utf-8-sig"
)

output_goias_total.to_csv(
    "outputs/projecao_total_goias_demo.csv",
    index=False,
    encoding="utf-8-sig"
)

output_by_project.to_csv(
    "outputs/contribuicao_projetos_demo.csv",
    index=False,
    encoding="utf-8-sig"
)

output_sensitivity_detail.to_csv(
    "outputs/sensibilidade_detalhada_demo.csv",
    index=False,
    encoding="utf-8-sig"
)

output_sensitivity_goias.to_csv(
    "outputs/sensibilidade_total_goias_demo.csv",
    index=False,
    encoding="utf-8-sig"
)

output_sensitivity_summary.to_csv(
    "outputs/sensibilidade_resumo_demo.csv",
    index=False,
    encoding="utf-8-sig"
)

# Cria um único arquivo ZIP com todos os outputs.
shutil.make_archive(
    "MINERA_Goias_outputs_demo_v2",
    "zip",
    "outputs"
)

print("Arquivos criados:")
print(os.listdir("outputs"))

files.download("MINERA_Goias_outputs_demo_v2.zip")

Arquivos criados:
['contribuicao_projetos_demo.csv', 'sensibilidade_detalhada_demo.csv', 'projecao_por_mineral_demo.csv', 'sensibilidade_total_goias_demo.csv', 'projecao_total_goias_demo.csv', 'sensibilidade_resumo_demo.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>